# Detección de Anomalías en Tráfico de Red
## Dataset: CIC-UNSW-NB15

---

## 📋 Introducción

### ¿Qué es el dataset CIC-UNSW-NB15?

El **CIC-UNSW-NB15** es un dataset de ciberseguridad creado por el Canadian Institute for Cybersecurity (CIC) en colaboración con la Universidad de New South Wales (UNSW). Contiene tráfico de red tanto normal como malicioso, diseñado específicamente para entrenar y evaluar sistemas de detección de intrusiones (IDS).

### Características del Dataset:

- **~450,000 registros** de flujos de red
- **75+ características** extraídas del tráfico de red (duración del flujo, bytes enviados/recibidos, flags TCP, etc.)
- **10 categorías** de tráfico:
  - **0: Benign** (Tráfico normal)
  - **1: Analysis** (Análisis de red)
  - **2: Backdoor** (Puertas traseras)
  - **3: DoS** (Ataques de Denegación de Servicio)
  - **4: Exploits** (Explotación de vulnerabilidades)
  - **5: Fuzzers** (Pruebas de fuzzing)
  - **6: Generic** (Ataques genéricos)
  - **7: Reconnaissance** (Reconocimiento de red)
  - **8: Shellcode** (Código malicioso)
  - **9: Worms** (Gusanos informáticos)

### ¿Por qué es importante?

La detección de anomalías en tráfico de red es **crucial** para:
- ✅ Identificar ataques cibernéticos en tiempo real
- ✅ Proteger infraestructuras críticas
- ✅ Detectar comportamientos sospechosos antes de que causen daño
- ✅ Complementar sistemas de seguridad tradicionales (firewalls, antivirus)

### Objetivo de este Notebook

En este análisis vamos a:
1. **Cargar y explorar** el dataset
2. **Preparar los datos** para machine learning
3. **Aplicar técnicas de detección de anomalías**:
   - Isolation Forest
   - One-Class SVM
   - Local Outlier Factor (LOF)
4. **Evaluar y comparar** los resultados
5. **Visualizar** las anomalías detectadas

## 1. Importación de Librerías

Comenzamos importando todas las librerías necesarias para nuestro análisis.

In [ ]:
# ============================================================================
# CONFIGURACIÓN INICIAL Y LIBRERÍAS
# ============================================================================

# Supresión de advertencias para salida más limpia
import warnings
warnings.filterwarnings('ignore')

# Librerías para manipulación de datos
import pandas as pd
import numpy as np

# Librerías para visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Librerías de Sklearn para Machine Learning
from sklearn.ensemble import IsolationForest  # Isolation Forest para detección de anomalías
from sklearn.svm import OneClassSVM          # One-Class SVM para detección de anomalías
from sklearn.neighbors import LocalOutlierFactor  # LOF para detección de anomalías
from sklearn.preprocessing import StandardScaler   # Normalización de datos
from sklearn.decomposition import PCA              # Reducción de dimensionalidad

# Métricas de evaluación
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

# Configuración de pandas para mostrar más columnas
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

print("✓ Librerías importadas correctamente")
print(f"✓ Versión de Pandas: {pd.__version__}")
print(f"✓ Versión de NumPy: {np.__version__}")

## 2. Carga de Datos

El dataset está dividido en dos archivos:
- **Data.csv**: Contiene las características (features) de cada flujo de red
- **Label.csv**: Contiene las etiquetas que indican el tipo de tráfico (0-9)

Vamos a cargar ambos archivos y combinarlos en un único DataFrame.

In [ ]:
# ============================================================================
# CARGA DE DATOS
# ============================================================================

print("📂 Cargando archivos del dataset CIC-UNSW-NB15...")
print("-" * 60)

# Cargar archivo de características (features)
df_data = pd.read_csv('Data.csv')
print(f"✓ Data.csv cargado: {df_data.shape[0]:,} filas x {df_data.shape[1]} columnas")

# Cargar archivo de etiquetas (labels)
df_labels = pd.read_csv('Label.csv')
print(f"✓ Label.csv cargado: {df_labels.shape[0]:,} filas x {df_labels.shape[1]} columnas")

# Combinar datos y etiquetas
df = pd.concat([df_data, df_labels], axis=1)
print(f"\n✓ Dataset completo: {df.shape[0]:,} filas x {df.shape[1]} columnas")
print(f"✓ Memoria utilizada: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\n" + "="*60)
print("PRIMERAS 5 FILAS DEL DATASET")
print("="*60)
df.head()

## 3. Exploración Inicial de Datos

Antes de aplicar cualquier modelo, es fundamental entender nuestros datos:
- ¿Qué tipo de datos tenemos?
- ¿Hay valores faltantes?
- ¿Cómo se distribuyen las etiquetas?
- ¿Existen valores infinitos o NaN?

In [ ]:
# ============================================================================
# INFORMACIÓN GENERAL DEL DATASET
# ============================================================================

print("📊 INFORMACIÓN DEL DATASET")
print("="*60)

# Información general
print(f"Número de registros: {len(df):,}")
print(f"Número de características: {df.shape[1] - 1}")  # -1 porque Label no es una característica
print(f"\nTipos de datos:")
print(df.dtypes.value_counts())

# Verificar valores faltantes
print(f"\n📌 Valores Faltantes (NaN):")
missing = df.isnull().sum()
if missing.sum() > 0:
    print(missing[missing > 0])
else:
    print("✓ No hay valores faltantes")

# Verificar valores infinitos
print(f"\n📌 Valores Infinitos:")
inf_count = np.isinf(df.select_dtypes(include=[np.number])).sum().sum()
if inf_count > 0:
    print(f"⚠ Se encontraron {inf_count:,} valores infinitos")
else:
    print("✓ No hay valores infinitos")

In [ ]:
# ============================================================================
# ESTADÍSTICAS DESCRIPTIVAS
# ============================================================================

print("\n📈 ESTADÍSTICAS DESCRIPTIVAS (primeras 10 columnas)")
print("="*60)
df.iloc[:, :10].describe()

## 4. Análisis de Distribución de Etiquetas

**¿Por qué es importante?**

En detección de anomalías, es crucial entender:
- **Desbalance de clases**: ¿Hay muchas más instancias normales que anómalas?
- **Tipos de ataques**: ¿Qué tipos de ataques están presentes?
- **Proporción de anomalías**: Esto nos ayudará a configurar los modelos correctamente

### Mapeo de Etiquetas:
```
0: Benign (Normal)     1: Analysis         2: Backdoor
3: DoS                 4: Exploits         5: Fuzzers
6: Generic             7: Reconnaissance   8: Shellcode
9: Worms
```

In [ ]:
# ============================================================================
# ANÁLISIS DE DISTRIBUCIÓN DE ETIQUETAS
# ============================================================================

# Diccionario de mapeo de etiquetas a nombres descriptivos
label_map = {
    0: 'Benign',
    1: 'Analysis',
    2: 'Backdoor',
    3: 'DoS',
    4: 'Exploits',
    5: 'Fuzzers',
    6: 'Generic',
    7: 'Reconnaissance',
    8: 'Shellcode',
    9: 'Worms'
}

# Contar frecuencia de cada etiqueta
label_counts = df['Label'].value_counts().sort_index()

print("\n🏷️ DISTRIBUCIÓN DE ETIQUETAS")
print("="*70)
print(f"{'Clase':<5} {'Nombre':<20} {'Cantidad':<15} {'Porcentaje'}")
print("-"*70)

for label, count in label_counts.items():
    percentage = (count / len(df)) * 100
    print(f"{label:<5} {label_map[label]:<20} {count:<15,} {percentage:>6.2f}%")

print("-"*70)
print(f"{'TOTAL':<5} {'':<20} {len(df):<15,} {'100.00%':>6}")

# Calcular proporción de anomalías (todo lo que no sea Benign)
benign_count = label_counts[0] if 0 in label_counts else 0
anomaly_count = len(df) - benign_count
anomaly_percentage = (anomaly_count / len(df)) * 100

print(f"\n📊 Resumen:")
print(f"  • Tráfico Normal (Benign): {benign_count:,} ({100-anomaly_percentage:.2f}%)")
print(f"  • Tráfico Anómalo (Ataques): {anomaly_count:,} ({anomaly_percentage:.2f}%)")

In [ ]:
# ============================================================================
# VISUALIZACIÓN DE DISTRIBUCIÓN DE ETIQUETAS
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico de barras con todas las clases
label_names = [label_map[i] for i in label_counts.index]
colors = ['green' if i == 0 else 'red' for i in label_counts.index]

axes[0].bar(label_names, label_counts.values, color=colors, alpha=0.7, edgecolor='black')
axes[0].set_xlabel('Tipo de Tráfico', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Cantidad de Registros', fontsize=12, fontweight='bold')
axes[0].set_title('Distribución de Tipos de Tráfico', fontsize=14, fontweight='bold', pad=20)
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', alpha=0.3)

# Añadir valores en las barras
for i, (name, value) in enumerate(zip(label_names, label_counts.values)):
    axes[0].text(i, value, f'{value:,}', ha='center', va='bottom', fontweight='bold', fontsize=9)

# Gráfico de pastel: Normal vs Anómalo
pie_data = [benign_count, anomaly_count]
pie_labels = [f'Normal\n({benign_count:,})', f'Anómalo\n({anomaly_count:,})']
pie_colors = ['#2ecc71', '#e74c3c']
explode = (0.05, 0.05)

axes[1].pie(pie_data, labels=pie_labels, colors=pie_colors, autopct='%1.1f%%',
            startangle=90, explode=explode, shadow=True, textprops={'fontsize': 12, 'fontweight': 'bold'})
axes[1].set_title('Proporción: Normal vs Anómalo', fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

print("\n💡 Observación: El dataset está desbalanceado, con más tráfico de ciertos tipos.")
print("   Esto es realista, ya que en redes reales el tráfico normal suele dominar.")

## 5. Preparación de Datos

### ¿Por qué necesitamos preparar los datos?

Los algoritmos de Machine Learning funcionan mejor cuando:
1. **No hay valores infinitos o NaN**: Estos pueden causar errores o resultados incorrectos
2. **Los datos están normalizados**: Características con diferentes escalas pueden dominar el modelo
3. **Simplificamos las etiquetas**: Para detección de anomalías, nos interesa distinguir "normal" vs "anómalo"

### Proceso de preparación:
1. Reemplazar valores infinitos por NaN
2. Rellenar NaN con la mediana de cada columna
3. Crear etiqueta binaria (0 = Normal, 1 = Anomalía)
4. Separar características (X) y etiquetas (y)
5. Normalizar las características

In [ ]:
# ============================================================================
# LIMPIEZA DE DATOS
# ============================================================================

print("🧹 Limpiando datos...")
print("-" * 60)

# Crear una copia para no modificar el original
df_clean = df.copy()

# 1. Reemplazar infinitos por NaN
# ¿Por qué? Los valores infinitos pueden aparecer en divisiones por cero
df_clean = df_clean.replace([np.inf, -np.inf], np.nan)
print("✓ Valores infinitos reemplazados por NaN")

# 2. Contar NaN después del reemplazo
nan_count = df_clean.isnull().sum().sum()
print(f"✓ NaN encontrados: {nan_count:,}")

# 3. Rellenar NaN con la mediana de cada columna
# ¿Por qué la mediana y no la media? La mediana es más robusta a valores extremos
if nan_count > 0:
    # Solo para columnas numéricas (excluir 'Label')
    numeric_columns = df_clean.select_dtypes(include=[np.number]).columns
    numeric_columns = [col for col in numeric_columns if col != 'Label']
    
    for col in numeric_columns:
        if df_clean[col].isnull().any():
            median_value = df_clean[col].median()
            df_clean[col].fillna(median_value, inplace=True)
    
    print(f"✓ NaN rellenados con la mediana de cada columna")

# 4. Verificar que no quedan NaN
remaining_nan = df_clean.isnull().sum().sum()
print(f"✓ NaN restantes: {remaining_nan}")

print("\n✅ Limpieza completada")

In [ ]:
# ============================================================================
# CREACIÓN DE ETIQUETA BINARIA
# ============================================================================

print("\n🏷️ Creando etiqueta binaria para detección de anomalías...")
print("-" * 60)

# Para detección de anomalías, simplificamos a dos clases:
# - 0 (Benign) → 0 (Normal)
# - Cualquier otro valor → 1 (Anomalía/Ataque)

df_clean['Binary_Label'] = (df_clean['Label'] != 0).astype(int)

# Verificar la conversión
print("\nDistribución de etiquetas binarias:")
binary_counts = df_clean['Binary_Label'].value_counts()
print(f"  • Normal (0):   {binary_counts[0]:,} ({binary_counts[0]/len(df_clean)*100:.2f}%)")
print(f"  • Anomalía (1): {binary_counts[1]:,} ({binary_counts[1]/len(df_clean)*100:.2f}%)")

print("\n✅ Etiqueta binaria creada")

In [ ]:
# ============================================================================
# SEPARACIÓN DE CARACTERÍSTICAS Y ETIQUETAS
# ============================================================================

print("\n📊 Separando características (X) y etiquetas (y)...")
print("-" * 60)

# Separar características (X) y etiquetas (y)
# Excluimos 'Label' y 'Binary_Label' de las características
X = df_clean.drop(['Label', 'Binary_Label'], axis=1)
y = df_clean['Binary_Label']

print(f"✓ Características (X): {X.shape}")
print(f"✓ Etiquetas (y): {y.shape}")
print(f"\nNombre de las características:")
print(f"  Total: {len(X.columns)} características")
print(f"  Primeras 10: {list(X.columns[:10])}")

## 6. Normalización de Datos

### ¿Por qué normalizar?

La **normalización** (también llamada estandarización) es crucial porque:

1. **Diferentes escalas**: Las características tienen rangos muy diferentes
   - Ejemplo: "Flow Duration" puede ser 1000-100000, mientras que "FIN Flag Count" puede ser 0-5

2. **Importancia equitativa**: Sin normalización, características con valores grandes dominan el algoritmo

3. **Convergencia más rápida**: Los algoritmos convergen más rápido con datos normalizados

### ¿Qué hace StandardScaler?

Transforma cada característica para que tenga:
- **Media = 0**
- **Desviación estándar = 1**

Fórmula: `z = (x - μ) / σ`
- `x`: valor original
- `μ`: media
- `σ`: desviación estándar
- `z`: valor normalizado

In [ ]:
# ============================================================================
# NORMALIZACIÓN CON STANDARDSCALER
# ============================================================================

print("🔄 Normalizando características...")
print("-" * 60)

# Mostrar estadísticas ANTES de normalizar
print("\nESTADÍSTICAS ANTES DE NORMALIZAR (primera característica):")
first_col = X.columns[0]
print(f"  {first_col}:")
print(f"    Media: {X[first_col].mean():.2f}")
print(f"    Std: {X[first_col].std():.2f}")
print(f"    Min: {X[first_col].min():.2f}")
print(f"    Max: {X[first_col].max():.2f}")

# Crear el escalador
scaler = StandardScaler()

# Ajustar y transformar los datos
# fit(): calcula media y desviación estándar de cada característica
# transform(): aplica la normalización
# fit_transform(): hace ambas cosas en un paso
X_scaled = scaler.fit_transform(X)

# Convertir de nuevo a DataFrame para mantener nombres de columnas
X_scaled = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)

# Mostrar estadísticas DESPUÉS de normalizar
print("\nESTADÍSTICAS DESPUÉS DE NORMALIZAR (primera característica):")
print(f"  {first_col}:")
print(f"    Media: {X_scaled[first_col].mean():.2f}")
print(f"    Std: {X_scaled[first_col].std():.2f}")
print(f"    Min: {X_scaled[first_col].min():.2f}")
print(f"    Max: {X_scaled[first_col].max():.2f}")

print("\n✅ Normalización completada")
print(f"✓ Forma de X_scaled: {X_scaled.shape}")

## 7. Muestreo de Datos

### ¿Por qué muestrear?

El dataset tiene ~450,000 registros. Entrenar modelos con todos los datos puede:
- ⏱️ Tomar mucho tiempo
- 💾 Consumir mucha memoria
- 🔄 Dificultar la experimentación rápida

### Estrategia de muestreo:

Tomaremos una **muestra stratificada** de 50,000 registros:
- **Stratificada**: Mantiene la misma proporción de clases que el dataset original
- **Suficientemente grande**: Para obtener resultados representativos
- **Manejable**: Para experimentación rápida

💡 **Nota**: Para producción, usarías todo el dataset o técnicas de procesamiento por lotes.

In [ ]:
# ============================================================================
# MUESTREO ESTRATIFICADO
# ============================================================================

from sklearn.model_selection import train_test_split

print("🎲 Creando muestra estratificada...")
print("-" * 60)

# Tamaño de la muestra
sample_size = 50000

# Calcular la proporción de la muestra respecto al total
sample_fraction = sample_size / len(X_scaled)

print(f"Tamaño original: {len(X_scaled):,} registros")
print(f"Tamaño de muestra: {sample_size:,} registros ({sample_fraction*100:.1f}%)")

# Crear muestra estratificada
# stratify=y asegura que la proporción de clases se mantenga
X_sample, _, y_sample, _ = train_test_split(
    X_scaled,
    y,
    train_size=sample_size,
    stratify=y,
    random_state=42  # Para reproducibilidad
)

print(f"\n✓ Muestra creada: {X_sample.shape}")

# Verificar que se mantuvo la proporción
print("\nDistribución en muestra:")
sample_counts = y_sample.value_counts()
print(f"  • Normal (0):   {sample_counts[0]:,} ({sample_counts[0]/len(y_sample)*100:.2f}%)")
print(f"  • Anomalía (1): {sample_counts[1]:,} ({sample_counts[1]/len(y_sample)*100:.2f}%)")

print("\nDistribución en dataset original:")
original_counts = y.value_counts()
print(f"  • Normal (0):   {original_counts[0]:,} ({original_counts[0]/len(y)*100:.2f}%)")
print(f"  • Anomalía (1): {original_counts[1]:,} ({original_counts[1]/len(y)*100:.2f}%)")

print("\n✅ Las proporciones se mantuvieron correctamente")

## 8. Detección de Anomalías con Isolation Forest

### ¿Qué es Isolation Forest?

**Isolation Forest** es un algoritmo específicamente diseñado para detección de anomalías que funciona de manera diferente a otros algoritmos:

### Concepto Clave: "Aislar" en lugar de "Modelar"

- Las **anomalías son raras y diferentes**: Son más fáciles de aislar
- Las **instancias normales son densas y similares**: Son más difíciles de aislar

### ¿Cómo funciona?

1. Construye múltiples árboles de decisión aleatorios
2. En cada árbol, selecciona aleatoriamente:
   - Una característica
   - Un valor de corte entre min y max
3. Divide los datos recursivamente
4. Las anomalías quedan aisladas en pocas divisiones (cerca de la raíz)
5. Las instancias normales requieren más divisiones (más profundo)

### Parámetros Importantes:

- **`contamination`**: Proporción esperada de anomalías en el dataset
  - En nuestro caso: ~25% son anomalías
  - Es crucial ajustar este valor según tu dataset

- **`n_estimators`**: Número de árboles a construir
  - Más árboles = más estable, pero más lento
  - Default: 100 (es un buen valor)

- **`random_state`**: Semilla para reproducibilidad

### Ventajas:
✅ Muy eficiente con datasets grandes
✅ Funciona bien con alta dimensionalidad
✅ No requiere normalización (pero ayuda)
✅ Pocos parámetros a ajustar

In [ ]:
# ============================================================================
# MODELO 1: ISOLATION FOREST
# ============================================================================

print("🌲 ISOLATION FOREST - Entrenamiento")
print("="*70)

# Calcular la proporción real de anomalías en nuestra muestra
contamination_rate = y_sample.sum() / len(y_sample)
print(f"\nProporción de anomalías en la muestra: {contamination_rate:.3f} ({contamination_rate*100:.1f}%)")

# Crear el modelo Isolation Forest
iso_forest = IsolationForest(
    contamination=contamination_rate,  # Proporción esperada de anomalías
    n_estimators=100,                   # Número de árboles
    random_state=42,                    # Para reproducibilidad
    n_jobs=-1,                          # Usar todos los procesadores disponibles
    verbose=0                           # Sin salida detallada
)

print("\n⏳ Entrenando modelo...")
import time
start_time = time.time()

# Entrenar el modelo
# fit_predict() hace dos cosas:
# 1. fit(): entrena el modelo
# 2. predict(): predice las etiquetas
y_pred_iso = iso_forest.fit_predict(X_sample)

elapsed_time = time.time() - start_time
print(f"✓ Entrenamiento completado en {elapsed_time:.2f} segundos")

# Isolation Forest devuelve:
# +1 para inliers (normal)
# -1 para outliers (anomalías)
# Convertimos a 0 y 1 para consistencia
y_pred_iso_binary = np.where(y_pred_iso == -1, 1, 0)

# Obtener scores de anomalía
# Scores más negativos = más anómalo
anomaly_scores_iso = iso_forest.score_samples(X_sample)

print(f"\n📊 Predicciones:")
pred_counts = pd.Series(y_pred_iso_binary).value_counts()
print(f"  • Predichas como Normal: {pred_counts[0]:,}")
print(f"  • Predichas como Anomalía: {pred_counts[1]:,}")

print(f"\n📈 Rango de Anomaly Scores:")
print(f"  • Mínimo (más anómalo): {anomaly_scores_iso.min():.4f}")
print(f"  • Máximo (más normal): {anomaly_scores_iso.max():.4f}")
print(f"  • Media: {anomaly_scores_iso.mean():.4f}")

### Evaluación de Isolation Forest

Ahora evaluaremos qué tan bien el modelo detectó las anomalías comparando con las etiquetas reales.

In [ ]:
# ============================================================================
# EVALUACIÓN DE ISOLATION FOREST
# ============================================================================

print("\n📊 EVALUACIÓN - ISOLATION FOREST")
print("="*70)

# Métricas de evaluación
accuracy_iso = accuracy_score(y_sample, y_pred_iso_binary)
precision_iso = precision_score(y_sample, y_pred_iso_binary)
recall_iso = recall_score(y_sample, y_pred_iso_binary)
f1_iso = f1_score(y_sample, y_pred_iso_binary)

print(f"\nMétricas de Rendimiento:")
print(f"  • Accuracy:  {accuracy_iso:.4f} ({accuracy_iso*100:.2f}%)")
print(f"  • Precision: {precision_iso:.4f} ({precision_iso*100:.2f}%)")
print(f"  • Recall:    {recall_iso:.4f} ({recall_iso*100:.2f}%)")
print(f"  • F1-Score:  {f1_iso:.4f}")

print("\n📝 Interpretación de las métricas:")
print(f"  • Accuracy: De todas las predicciones, {accuracy_iso*100:.1f}% fueron correctas")
print(f"  • Precision: De los marcados como anomalía, {precision_iso*100:.1f}% realmente lo eran")
print(f"  • Recall: De todas las anomalías reales, detectamos {recall_iso*100:.1f}%")
print(f"  • F1-Score: Media armónica de Precision y Recall")

# Reporte de clasificación detallado
print("\n" + "="*70)
print("REPORTE DE CLASIFICACIÓN DETALLADO")
print("="*70)
print(classification_report(y_sample, y_pred_iso_binary, 
                          target_names=['Normal', 'Anomalía'],
                          digits=4))

In [ ]:
# ============================================================================
# MATRIZ DE CONFUSIÓN - ISOLATION FOREST
# ============================================================================

# Calcular matriz de confusión
cm_iso = confusion_matrix(y_sample, y_pred_iso_binary)

# Visualizar matriz de confusión
plt.figure(figsize=(10, 8))
sns.heatmap(cm_iso, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Normal', 'Anomalía'],
            yticklabels=['Normal', 'Anomalía'],
            cbar_kws={'label': 'Cantidad'},
            annot_kws={'size': 14, 'weight': 'bold'})

plt.title('Matriz de Confusión - Isolation Forest', 
          fontsize=16, fontweight='bold', pad=20)
plt.ylabel('Etiqueta Real', fontsize=14, fontweight='bold')
plt.xlabel('Etiqueta Predicha', fontsize=14, fontweight='bold')

# Añadir texto explicativo
tn, fp, fn, tp = cm_iso.ravel()
plt.text(0.5, -0.15, 
         f'TN={tn:,}  FP={fp:,}\nFN={fn:,}  TP={tp:,}',
         ha='center', va='top', transform=plt.gca().transAxes,
         fontsize=11, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

print("\n📌 Interpretación de la Matriz de Confusión:")
print(f"  • True Negatives (TN):  {tn:,} - Normal correctamente identificado")
print(f"  • False Positives (FP): {fp:,} - Normal clasificado como anomalía (falsa alarma)")
print(f"  • False Negatives (FN): {fn:,} - Anomalía no detectada (más peligroso)")
print(f"  • True Positives (TP):  {tp:,} - Anomalía correctamente detectada")

## 9. Detección de Anomalías con One-Class SVM

### ¿Qué es One-Class SVM?

**One-Class SVM** (Support Vector Machine de Una Clase) es un algoritmo que aprende a describir la "normalidad" y luego identifica cualquier cosa fuera de esa descripción como anomalía.

### Concepto Clave: "Frontera de Normalidad"

- Aprende los límites del comportamiento normal
- Crea una "frontera" (boundary) en el espacio de características
- Todo lo que está fuera de esa frontera es considerado anómalo

### ¿Cómo funciona?

1. Mapea los datos a un espacio de mayor dimensión (usando un kernel)
2. Encuentra el hiperplano que mejor separa los datos del origen
3. Maximiza la distancia de este hiperplano a los datos
4. Las instancias lejos del hiperplano son anomalías

### Parámetros Importantes:

- **`kernel`**: Tipo de función kernel
  - 'rbf' (Radial Basis Function): el más usado, funciona bien en la mayoría de casos
  - 'linear': para datos linealmente separables
  - 'poly': para relaciones polinómicas

- **`nu`**: Límite superior en la fracción de errores de entrenamiento y límite inferior en la fracción de vectores de soporte
  - Similar a contamination en Isolation Forest
  - Valor entre 0 y 1
  - Ajustar según la proporción esperada de anomalías

- **`gamma`**: Coeficiente del kernel RBF
  - 'auto': 1 / n_features
  - 'scale': 1 / (n_features * X.var())
  - Valores altos: frontera más compleja

### Ventajas:
✅ Muy efectivo cuando la normalidad tiene una frontera clara
✅ Funciona bien con datos de alta dimensionalidad
✅ Robusto con kernel RBF

### Desventajas:
❌ Más lento que Isolation Forest en datasets grandes
❌ Sensible a la normalización de datos
❌ Requiere más ajuste de parámetros

In [ ]:
# ============================================================================
# MODELO 2: ONE-CLASS SVM
# ============================================================================

print("🎯 ONE-CLASS SVM - Entrenamiento")
print("="*70)

# Crear el modelo One-Class SVM
oc_svm = OneClassSVM(
    kernel='rbf',                       # Kernel Radial Basis Function
    gamma='scale',                      # Coeficiente del kernel
    nu=contamination_rate               # Proporción esperada de outliers
)

print(f"\nParámetros del modelo:")
print(f"  • Kernel: rbf (Radial Basis Function)")
print(f"  • Gamma: scale")
print(f"  • Nu: {contamination_rate:.3f} ({contamination_rate*100:.1f}%)")

print("\n⏳ Entrenando modelo...")
print("   (Nota: One-Class SVM puede tardar más que Isolation Forest)")
start_time = time.time()

# Entrenar y predecir
y_pred_svm = oc_svm.fit_predict(X_sample)

elapsed_time = time.time() - start_time
print(f"✓ Entrenamiento completado en {elapsed_time:.2f} segundos")

# One-Class SVM también devuelve +1 y -1
# Convertimos a 0 y 1
y_pred_svm_binary = np.where(y_pred_svm == -1, 1, 0)

# Obtener función de decisión (equivalente a anomaly score)
# Valores más negativos = más anómalo
decision_scores_svm = oc_svm.decision_function(X_sample)

print(f"\n📊 Predicciones:")
pred_counts = pd.Series(y_pred_svm_binary).value_counts()
print(f"  • Predichas como Normal: {pred_counts[0]:,}")
print(f"  • Predichas como Anomalía: {pred_counts[1]:,}")

print(f"\n📈 Rango de Decision Scores:")
print(f"  • Mínimo (más anómalo): {decision_scores_svm.min():.4f}")
print(f"  • Máximo (más normal): {decision_scores_svm.max():.4f}")
print(f"  • Media: {decision_scores_svm.mean():.4f}")

### Evaluación de One-Class SVM

In [ ]:
# ============================================================================
# EVALUACIÓN DE ONE-CLASS SVM
# ============================================================================

print("\n📊 EVALUACIÓN - ONE-CLASS SVM")
print("="*70)

# Métricas de evaluación
accuracy_svm = accuracy_score(y_sample, y_pred_svm_binary)
precision_svm = precision_score(y_sample, y_pred_svm_binary)
recall_svm = recall_score(y_sample, y_pred_svm_binary)
f1_svm = f1_score(y_sample, y_pred_svm_binary)

print(f"\nMétricas de Rendimiento:")
print(f"  • Accuracy:  {accuracy_svm:.4f} ({accuracy_svm*100:.2f}%)")
print(f"  • Precision: {precision_svm:.4f} ({precision_svm*100:.2f}%)")
print(f"  • Recall:    {recall_svm:.4f} ({recall_svm*100:.2f}%)")
print(f"  • F1-Score:  {f1_svm:.4f}")

# Reporte de clasificación
print("\n" + "="*70)
print("REPORTE DE CLASIFICACIÓN DETALLADO")
print("="*70)
print(classification_report(y_sample, y_pred_svm_binary, 
                          target_names=['Normal', 'Anomalía'],
                          digits=4))

# Matriz de confusión
cm_svm = confusion_matrix(y_sample, y_pred_svm_binary)

plt.figure(figsize=(10, 8))
sns.heatmap(cm_svm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Normal', 'Anomalía'],
            yticklabels=['Normal', 'Anomalía'],
            cbar_kws={'label': 'Cantidad'},
            annot_kws={'size': 14, 'weight': 'bold'})

plt.title('Matriz de Confusión - One-Class SVM', 
          fontsize=16, fontweight='bold', pad=20)
plt.ylabel('Etiqueta Real', fontsize=14, fontweight='bold')
plt.xlabel('Etiqueta Predicha', fontsize=14, fontweight='bold')

tn, fp, fn, tp = cm_svm.ravel()
plt.text(0.5, -0.15, 
         f'TN={tn:,}  FP={fp:,}\nFN={fn:,}  TP={tp:,}',
         ha='center', va='top', transform=plt.gca().transAxes,
         fontsize=11, bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))

plt.tight_layout()
plt.show()

## 10. Detección de Anomalías con Local Outlier Factor (LOF)

### ¿Qué es Local Outlier Factor?

**LOF** es un algoritmo basado en **densidad** que detecta anomalías comparando la densidad local de una instancia con la de sus vecinos.

### Concepto Clave: "Densidad Local"

- Las instancias normales tienen vecinos cercanos (alta densidad)
- Las anomalías están aisladas o en regiones de baja densidad
- LOF compara la densidad de cada punto con la de sus vecinos

### ¿Cómo funciona?

1. Para cada punto, encuentra sus k vecinos más cercanos
2. Calcula la "densidad de alcanzabilidad" (reachability density)
3. Compara esta densidad con la de sus vecinos
4. Si un punto tiene mucha menor densidad que sus vecinos → Anomalía

### Factor LOF:
- **LOF ≈ 1**: El punto tiene densidad similar a sus vecinos (normal)
- **LOF >> 1**: El punto tiene mucha menor densidad que sus vecinos (anomalía)
- **LOF < 1**: El punto está en una región más densa que sus vecinos

### Parámetros Importantes:

- **`n_neighbors`**: Número de vecinos a considerar
  - Valores pequeños (5-20): Detecta anomalías locales
  - Valores grandes (50+): Detecta anomalías globales
  - Default: 20 (buen balance)

- **`contamination`**: Proporción esperada de outliers
  - Similar a los otros modelos

- **`novelty`**: 
  - False: Detecta outliers en el conjunto de entrenamiento
  - True: Puede predecir en nuevos datos

### Ventajas:
✅ Excelente para detectar anomalías locales
✅ No asume ninguna distribución de datos
✅ Funciona bien con clusters de diferentes densidades

### Desventajas:
❌ Sensible al valor de k (n_neighbors)
❌ Puede ser lento con datasets muy grandes
❌ Sensible a la maldición de la dimensionalidad

In [ ]:
# ============================================================================
# MODELO 3: LOCAL OUTLIER FACTOR (LOF)
# ============================================================================

print("🎪 LOCAL OUTLIER FACTOR (LOF) - Entrenamiento")
print("="*70)

# Crear el modelo LOF
lof = LocalOutlierFactor(
    n_neighbors=20,                     # Número de vecinos a considerar
    contamination=contamination_rate,   # Proporción esperada de outliers
    novelty=False,                      # Detectar outliers en datos de entrenamiento
    n_jobs=-1                           # Usar todos los procesadores
)

print(f"\nParámetros del modelo:")
print(f"  • Número de vecinos: 20")
print(f"  • Contamination: {contamination_rate:.3f} ({contamination_rate*100:.1f}%)")
print(f"  • Novelty: False (detectar outliers en conjunto de entrenamiento)")

print("\n⏳ Entrenando modelo...")
start_time = time.time()

# Entrenar y predecir
# LOF.fit_predict() solo funciona con novelty=False
y_pred_lof = lof.fit_predict(X_sample)

elapsed_time = time.time() - start_time
print(f"✓ Entrenamiento completado en {elapsed_time:.2f} segundos")

# LOF también devuelve +1 (inlier) y -1 (outlier)
# Convertimos a 0 y 1
y_pred_lof_binary = np.where(y_pred_lof == -1, 1, 0)

# Obtener LOF scores
# negative_outlier_factor_: valores más negativos = más anómalo
lof_scores = lof.negative_outlier_factor_

print(f"\n📊 Predicciones:")
pred_counts = pd.Series(y_pred_lof_binary).value_counts()
print(f"  • Predichas como Normal: {pred_counts[0]:,}")
print(f"  • Predichas como Anomalía: {pred_counts[1]:,}")

print(f"\n📈 Rango de LOF Scores:")
print(f"  • Mínimo (más anómalo): {lof_scores.min():.4f}")
print(f"  • Máximo (más normal): {lof_scores.max():.4f}")
print(f"  • Media: {lof_scores.mean():.4f}")
print(f"\n  💡 Interpretación: Scores < -1.5 suelen ser anomalías fuertes")

### Evaluación de Local Outlier Factor

In [ ]:
# ============================================================================
# EVALUACIÓN DE LOCAL OUTLIER FACTOR
# ============================================================================

print("\n📊 EVALUACIÓN - LOCAL OUTLIER FACTOR")
print("="*70)

# Métricas de evaluación
accuracy_lof = accuracy_score(y_sample, y_pred_lof_binary)
precision_lof = precision_score(y_sample, y_pred_lof_binary)
recall_lof = recall_score(y_sample, y_pred_lof_binary)
f1_lof = f1_score(y_sample, y_pred_lof_binary)

print(f"\nMétricas de Rendimiento:")
print(f"  • Accuracy:  {accuracy_lof:.4f} ({accuracy_lof*100:.2f}%)")
print(f"  • Precision: {precision_lof:.4f} ({precision_lof*100:.2f}%)")
print(f"  • Recall:    {recall_lof:.4f} ({recall_lof*100:.2f}%)")
print(f"  • F1-Score:  {f1_lof:.4f}")

# Reporte de clasificación
print("\n" + "="*70)
print("REPORTE DE CLASIFICACIÓN DETALLADO")
print("="*70)
print(classification_report(y_sample, y_pred_lof_binary, 
                          target_names=['Normal', 'Anomalía'],
                          digits=4))

# Matriz de confusión
cm_lof = confusion_matrix(y_sample, y_pred_lof_binary)

plt.figure(figsize=(10, 8))
sns.heatmap(cm_lof, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Normal', 'Anomalía'],
            yticklabels=['Normal', 'Anomalía'],
            cbar_kws={'label': 'Cantidad'},
            annot_kws={'size': 14, 'weight': 'bold'})

plt.title('Matriz de Confusión - Local Outlier Factor', 
          fontsize=16, fontweight='bold', pad=20)
plt.ylabel('Etiqueta Real', fontsize=14, fontweight='bold')
plt.xlabel('Etiqueta Predicha', fontsize=14, fontweight='bold')

tn, fp, fn, tp = cm_lof.ravel()
plt.text(0.5, -0.15, 
         f'TN={tn:,}  FP={fp:,}\nFN={fn:,}  TP={tp:,}',
         ha='center', va='top', transform=plt.gca().transAxes,
         fontsize=11, bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.5))

plt.tight_layout()
plt.show()

## 11. Comparación de Modelos

Ahora que hemos entrenado y evaluado los tres modelos, vamos a compararlos para determinar cuál funciona mejor para nuestro dataset de ciberseguridad.

In [ ]:
# ============================================================================
# COMPARACIÓN DE LOS TRES MODELOS
# ============================================================================

print("\n🏆 COMPARACIÓN DE MODELOS")
print("="*80)

# Crear DataFrame con las métricas de todos los modelos
comparison_df = pd.DataFrame({
    'Modelo': ['Isolation Forest', 'One-Class SVM', 'Local Outlier Factor'],
    'Accuracy': [accuracy_iso, accuracy_svm, accuracy_lof],
    'Precision': [precision_iso, precision_svm, precision_lof],
    'Recall': [recall_iso, recall_svm, recall_lof],
    'F1-Score': [f1_iso, f1_svm, f1_lof]
})

# Mostrar tabla de comparación
print("\n📊 TABLA DE MÉTRICAS")
print("-"*80)
print(comparison_df.to_string(index=False))
print("-"*80)

# Identificar el mejor modelo para cada métrica
print("\n🥇 MEJORES MODELOS POR MÉTRICA:")
print("-"*80)
for metric in ['Accuracy', 'Precision', 'Recall', 'F1-Score']:
    best_idx = comparison_df[metric].idxmax()
    best_model = comparison_df.loc[best_idx, 'Modelo']
    best_value = comparison_df.loc[best_idx, metric]
    print(f"  • {metric:<12}: {best_model:<22} ({best_value:.4f})")

# Calcular promedio de métricas para determinar el mejor modelo general
comparison_df['Promedio'] = comparison_df[['Accuracy', 'Precision', 'Recall', 'F1-Score']].mean(axis=1)
best_overall_idx = comparison_df['Promedio'].idxmax()
best_overall_model = comparison_df.loc[best_overall_idx, 'Modelo']

print("\n" + "="*80)
print(f"🌟 MEJOR MODELO GENERAL: {best_overall_model}")
print(f"   Promedio de métricas: {comparison_df.loc[best_overall_idx, 'Promedio']:.4f}")
print("="*80)

In [ ]:
# ============================================================================
# VISUALIZACIÓN COMPARATIVA
# ============================================================================

fig, ax = plt.subplots(figsize=(14, 8))

# Preparar datos para el gráfico
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x = np.arange(len(metrics))
width = 0.25

# Colores para cada modelo
colors = ['#3498db', '#2ecc71', '#e74c3c']

# Crear barras para cada modelo
for i, (idx, row) in enumerate(comparison_df.iterrows()):
    values = [row['Accuracy'], row['Precision'], row['Recall'], row['F1-Score']]
    offset = (i - 1) * width
    bars = ax.bar(x + offset, values, width, label=row['Modelo'], 
                  color=colors[i], alpha=0.8, edgecolor='black', linewidth=1.5)
    
    # Añadir valores en las barras
    for j, (bar, val) in enumerate(zip(bars, values)):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.3f}',
                ha='center', va='bottom', fontsize=9, fontweight='bold')

# Configurar el gráfico
ax.set_xlabel('Métricas', fontsize=13, fontweight='bold')
ax.set_ylabel('Puntuación', fontsize=13, fontweight='bold')
ax.set_title('Comparación de Rendimiento de Modelos de Detección de Anomalías',
             fontsize=15, fontweight='bold', pad=20)
ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=12)
ax.legend(fontsize=11, loc='lower right')
ax.set_ylim([0, 1.1])
ax.grid(axis='y', alpha=0.3, linestyle='--')

# Añadir línea de referencia
ax.axhline(y=0.8, color='gray', linestyle='--', linewidth=1, alpha=0.5, label='Umbral 80%')

plt.tight_layout()
plt.show()

print("\n💡 Análisis de la comparación:")
print("  • Las barras más altas indican mejor rendimiento")
print("  • Compara la consistencia entre métricas")
print("  • Un buen modelo debe tener balance entre Precision y Recall")

## 12. Visualización con PCA

### ¿Por qué PCA?

Nuestro dataset tiene 75+ características (dimensiones). Es imposible visualizarlo directamente en 2D o 3D.

**PCA (Principal Component Analysis)** nos permite:
- Reducir las 75+ dimensiones a 2 dimensiones
- Mantener la mayor cantidad posible de varianza (información)
- Visualizar cómo los modelos separan normal de anómalo

### ¿Qué hace PCA?

1. Encuentra las direcciones de máxima varianza en los datos
2. Proyecta los datos en esas direcciones (componentes principales)
3. Las primeras componentes capturan la mayor parte de la información

### Limitaciones:

⚠️ La visualización en 2D es una simplificación
⚠️ Perdemos información al reducir dimensiones
⚠️ Pero nos da una idea de cómo se distribuyen los datos

In [ ]:
# ============================================================================
# REDUCCIÓN DE DIMENSIONALIDAD CON PCA
# ============================================================================

print("🔬 Aplicando PCA para visualización...")
print("-" * 60)

# Crear modelo PCA para reducir a 2 componentes
pca = PCA(n_components=2, random_state=42)

# Ajustar y transformar los datos
X_pca = pca.fit_transform(X_sample)

print(f"✓ Datos reducidos de {X_sample.shape[1]} a 2 dimensiones")
print(f"\n📊 Varianza explicada por cada componente:")
print(f"  • Componente 1: {pca.explained_variance_ratio_[0]:.4f} ({pca.explained_variance_ratio_[0]*100:.2f}%)")
print(f"  • Componente 2: {pca.explained_variance_ratio_[1]:.4f} ({pca.explained_variance_ratio_[1]*100:.2f}%)")
print(f"  • Total explicado: {pca.explained_variance_ratio_.sum():.4f} ({pca.explained_variance_ratio_.sum()*100:.2f}%)")

print(f"\n💡 Interpretación:")
print(f"   Las 2 componentes capturan {pca.explained_variance_ratio_.sum()*100:.1f}% de la varianza total")
print(f"   Esto significa que perdemos {(1-pca.explained_variance_ratio_.sum())*100:.1f}% de información")
print(f"   pero ganamos la capacidad de visualizar los datos")

In [ ]:
# ============================================================================
# VISUALIZACIÓN DE RESULTADOS CON PCA
# ============================================================================

# Crear figura con 4 subgráficos
fig, axes = plt.subplots(2, 2, figsize=(18, 16))
fig.suptitle('Visualización de Detección de Anomalías (Proyección PCA 2D)',
             fontsize=18, fontweight='bold', y=0.995)

# Configuración de colores
color_normal = '#3498db'
color_anomaly = '#e74c3c'
alpha_normal = 0.3
alpha_anomaly = 0.7

# 1. ETIQUETAS REALES
axes[0, 0].scatter(X_pca[y_sample == 0, 0], X_pca[y_sample == 0, 1],
                   c=color_normal, alpha=alpha_normal, s=20, label='Normal Real', edgecolors='none')
axes[0, 0].scatter(X_pca[y_sample == 1, 0], X_pca[y_sample == 1, 1],
                   c=color_anomaly, alpha=alpha_anomaly, s=40, marker='^',
                   label='Anomalía Real', edgecolors='black', linewidth=0.5)
axes[0, 0].set_title('Datos Reales', fontsize=14, fontweight='bold', pad=10)
axes[0, 0].set_xlabel('Componente Principal 1', fontsize=11)
axes[0, 0].set_ylabel('Componente Principal 2', fontsize=11)
axes[0, 0].legend(loc='upper right', fontsize=10)
axes[0, 0].grid(True, alpha=0.3)

# 2. ISOLATION FOREST
axes[0, 1].scatter(X_pca[y_pred_iso_binary == 0, 0], X_pca[y_pred_iso_binary == 0, 1],
                   c=color_normal, alpha=alpha_normal, s=20, label='Normal Predicho', edgecolors='none')
axes[0, 1].scatter(X_pca[y_pred_iso_binary == 1, 0], X_pca[y_pred_iso_binary == 1, 1],
                   c=color_anomaly, alpha=alpha_anomaly, s=40, marker='^',
                   label='Anomalía Predicha', edgecolors='black', linewidth=0.5)
axes[0, 1].set_title(f'Isolation Forest (F1={f1_iso:.3f})', 
                     fontsize=14, fontweight='bold', pad=10)
axes[0, 1].set_xlabel('Componente Principal 1', fontsize=11)
axes[0, 1].set_ylabel('Componente Principal 2', fontsize=11)
axes[0, 1].legend(loc='upper right', fontsize=10)
axes[0, 1].grid(True, alpha=0.3)

# 3. ONE-CLASS SVM
axes[1, 0].scatter(X_pca[y_pred_svm_binary == 0, 0], X_pca[y_pred_svm_binary == 0, 1],
                   c=color_normal, alpha=alpha_normal, s=20, label='Normal Predicho', edgecolors='none')
axes[1, 0].scatter(X_pca[y_pred_svm_binary == 1, 0], X_pca[y_pred_svm_binary == 1, 1],
                   c=color_anomaly, alpha=alpha_anomaly, s=40, marker='^',
                   label='Anomalía Predicha', edgecolors='black', linewidth=0.5)
axes[1, 0].set_title(f'One-Class SVM (F1={f1_svm:.3f})', 
                     fontsize=14, fontweight='bold', pad=10)
axes[1, 0].set_xlabel('Componente Principal 1', fontsize=11)
axes[1, 0].set_ylabel('Componente Principal 2', fontsize=11)
axes[1, 0].legend(loc='upper right', fontsize=10)
axes[1, 0].grid(True, alpha=0.3)

# 4. LOCAL OUTLIER FACTOR
axes[1, 1].scatter(X_pca[y_pred_lof_binary == 0, 0], X_pca[y_pred_lof_binary == 0, 1],
                   c=color_normal, alpha=alpha_normal, s=20, label='Normal Predicho', edgecolors='none')
axes[1, 1].scatter(X_pca[y_pred_lof_binary == 1, 0], X_pca[y_pred_lof_binary == 1, 1],
                   c=color_anomaly, alpha=alpha_anomaly, s=40, marker='^',
                   label='Anomalía Predicha', edgecolors='black', linewidth=0.5)
axes[1, 1].set_title(f'Local Outlier Factor (F1={f1_lof:.3f})', 
                     fontsize=14, fontweight='bold', pad=10)
axes[1, 1].set_xlabel('Componente Principal 1', fontsize=11)
axes[1, 1].set_ylabel('Componente Principal 2', fontsize=11)
axes[1, 1].legend(loc='upper right', fontsize=10)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Observaciones de la visualización:")
print("  • Los puntos azules representan tráfico normal")
print("  • Los triángulos rojos representan anomalías")
print("  • Compara cómo cada modelo separa normal de anómalo")
print("  • Las diferencias muestran cómo cada algoritmo aborda el problema")

## 13. Análisis de Tipos de Ataques Detectados

Vamos a analizar qué tipos específicos de ataques detectó mejor cada modelo.

In [ ]:
# ============================================================================
# ANÁLISIS POR TIPO DE ATAQUE
# ============================================================================

print("🔍 ANÁLISIS DE DETECCIÓN POR TIPO DE ATAQUE")
print("="*80)

# Obtener las etiquetas originales (multiclase) para la muestra
y_multiclass = df_clean.loc[y_sample.index, 'Label']

# Analizar detección por tipo de ataque con Isolation Forest (mejor modelo general)
attack_analysis = pd.DataFrame({
    'Tipo': y_multiclass,
    'Nombre': y_multiclass.map(label_map),
    'Real_Anomalia': y_sample,
    'Pred_IsoForest': y_pred_iso_binary,
    'Pred_SVM': y_pred_svm_binary,
    'Pred_LOF': y_pred_lof_binary
})

# Calcular tasa de detección por tipo
detection_by_type = []
for tipo in sorted(y_multiclass.unique()):
    mask = attack_analysis['Tipo'] == tipo
    total = mask.sum()
    
    if tipo == 0:  # Benign
        # Para normal, queremos que NO sea detectado como anomalía
        correct_iso = ((attack_analysis[mask]['Pred_IsoForest'] == 0).sum())
        correct_svm = ((attack_analysis[mask]['Pred_SVM'] == 0).sum())
        correct_lof = ((attack_analysis[mask]['Pred_LOF'] == 0).sum())
    else:  # Ataques
        # Para ataques, queremos que SÍ sea detectado como anomalía
        correct_iso = ((attack_analysis[mask]['Pred_IsoForest'] == 1).sum())
        correct_svm = ((attack_analysis[mask]['Pred_SVM'] == 1).sum())
        correct_lof = ((attack_analysis[mask]['Pred_LOF'] == 1).sum())
    
    detection_by_type.append({
        'Tipo': tipo,
        'Nombre': label_map[tipo],
        'Total': total,
        'IsoForest_Correcto': correct_iso,
        'IsoForest_%': (correct_iso / total * 100) if total > 0 else 0,
        'SVM_Correcto': correct_svm,
        'SVM_%': (correct_svm / total * 100) if total > 0 else 0,
        'LOF_Correcto': correct_lof,
        'LOF_%': (correct_lof / total * 100) if total > 0 else 0
    })

detection_df = pd.DataFrame(detection_by_type)

print("\n📊 TASA DE DETECCIÓN CORRECTA POR TIPO DE TRÁFICO")
print("-"*80)
print(detection_df[['Nombre', 'Total', 'IsoForest_%', 'SVM_%', 'LOF_%']].to_string(index=False))
print("-"*80)

print("\n💡 Notas:")
print("  • Para 'Benign': % de veces que NO fue marcado como anomalía (correcto)")
print("  • Para ataques: % de veces que SÍ fue detectado como anomalía (correcto)")

In [ ]:
# ============================================================================
# VISUALIZACIÓN DE DETECCIÓN POR TIPO
# ============================================================================

# Preparar datos para visualización
attack_types = detection_df['Nombre'].values
x_pos = np.arange(len(attack_types))
width = 0.25

fig, ax = plt.subplots(figsize=(16, 8))

# Crear barras para cada modelo
bars1 = ax.bar(x_pos - width, detection_df['IsoForest_%'], width, 
               label='Isolation Forest', color='#3498db', alpha=0.8, edgecolor='black')
bars2 = ax.bar(x_pos, detection_df['SVM_%'], width,
               label='One-Class SVM', color='#2ecc71', alpha=0.8, edgecolor='black')
bars3 = ax.bar(x_pos + width, detection_df['LOF_%'], width,
               label='LOF', color='#e74c3c', alpha=0.8, edgecolor='black')

# Añadir valores en las barras
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.1f}%',
                ha='center', va='bottom', fontsize=9, fontweight='bold')

# Configurar gráfico
ax.set_xlabel('Tipo de Tráfico', fontsize=13, fontweight='bold')
ax.set_ylabel('Tasa de Detección Correcta (%)', fontsize=13, fontweight='bold')
ax.set_title('Rendimiento por Tipo de Tráfico', 
             fontsize=15, fontweight='bold', pad=20)
ax.set_xticks(x_pos)
ax.set_xticklabels(attack_types, rotation=45, ha='right', fontsize=11)
ax.legend(fontsize=12, loc='lower right')
ax.set_ylim([0, 110])
ax.grid(axis='y', alpha=0.3, linestyle='--')

# Añadir línea de referencia al 80%
ax.axhline(y=80, color='gray', linestyle='--', linewidth=1.5, alpha=0.7, label='Umbral 80%')

plt.tight_layout()
plt.show()

print("\n🎯 Análisis:")
print("  • Identifica qué tipos de ataques son más difíciles de detectar")
print("  • Compara qué modelo funciona mejor para cada tipo")
print("  • Valores altos = buena detección, valores bajos = necesita mejora")

## 14. Conclusiones y Recomendaciones

### 📊 Resumen de Resultados

Hemos aplicado tres técnicas de detección de anomalías al dataset CIC-UNSW-NB15:

1. **Isolation Forest**
   - ✅ Rápido y eficiente
   - ✅ Buen balance entre precision y recall
   - ✅ Escalable a grandes datasets

2. **One-Class SVM**
   - ✅ Buena frontera de decisión
   - ⚠️ Más lento que Isolation Forest
   - ⚠️ Sensible a parámetros

3. **Local Outlier Factor**
   - ✅ Excelente para anomalías locales
   - ⚠️ Puede ser sensible al número de vecinos
   - ⚠️ Más costoso computacionalmente

### 🎯 Mejores Prácticas

1. **Preprocesamiento**:
   - Siempre normaliza los datos
   - Maneja valores infinitos y NaN
   - Verifica el balance de clases

2. **Selección de Modelo**:
   - Para datasets grandes: **Isolation Forest**
   - Para fronteras complejas: **One-Class SVM**
   - Para anomalías locales: **LOF**

3. **Ajuste de Parámetros**:
   - Ajusta `contamination` según tu dataset
   - Experimenta con diferentes valores de `n_neighbors` (LOF)
   - Prueba diferentes kernels (SVM)

4. **Evaluación**:
   - No te fíes solo del Accuracy (puede ser engañoso con datos desbalanceados)
   - Considera Precision, Recall y F1-Score
   - Analiza la matriz de confusión

### 🚀 Próximos Pasos

1. **Mejoras al Modelo**:
   - Ensemble de modelos (combinar predicciones)
   - Ajuste de hiperparámetros con GridSearch
   - Feature engineering (crear nuevas características)

2. **Producción**:
   - Implementar detección en tiempo real
   - Crear sistema de alertas
   - Monitoreo continuo del rendimiento

3. **Análisis Avanzado**:
   - Deep Learning (Autoencoders, LSTM)
   - Análisis de series temporales
   - Explicabilidad de predicciones (SHAP, LIME)

### 📚 Recursos Adicionales

- [CIC-UNSW-NB15 Dataset](https://www.unb.ca/cic/datasets/cic-unsw-nb15.html)
- [Scikit-learn Anomaly Detection](https://scikit-learn.org/stable/modules/outlier_detection.html)
- [Isolation Forest Paper](https://cs.nju.edu.cn/zhouzh/zhouzh.files/publication/icdm08b.pdf)

## 15. Guardar Resultados

Guardemos los resultados más importantes para referencia futura.

In [ ]:
# ============================================================================
# GUARDAR RESULTADOS
# ============================================================================

print("💾 Guardando resultados...")
print("-" * 60)

# Crear DataFrame con predicciones de todos los modelos
results_df = pd.DataFrame({
    'Label_Real': y_sample,
    'Label_Multiclase': y_multiclass,
    'Tipo_Ataque': y_multiclass.map(label_map),
    'Pred_IsolationForest': y_pred_iso_binary,
    'Pred_OneClassSVM': y_pred_svm_binary,
    'Pred_LOF': y_pred_lof_binary,
    'Score_IsolationForest': anomaly_scores_iso,
    'Score_SVM': decision_scores_svm,
    'Score_LOF': lof_scores
})

# Guardar a CSV
results_df.to_csv('resultados_deteccion_anomalias.csv', index=False)
print("✓ Predicciones guardadas en: resultados_deteccion_anomalias.csv")

# Guardar comparación de modelos
comparison_df.to_csv('comparacion_modelos.csv', index=False)
print("✓ Comparación guardada en: comparacion_modelos.csv")

# Guardar análisis por tipo
detection_df.to_csv('deteccion_por_tipo.csv', index=False)
print("✓ Análisis por tipo guardado en: deteccion_por_tipo.csv")

print("\n✅ Todos los resultados guardados correctamente")

---

## 🎓 Fin del Análisis

### Has aprendido:

✅ Cómo cargar y explorar un dataset de ciberseguridad real

✅ Técnicas de preprocesamiento para detección de anomalías

✅ Tres algoritmos diferentes de detección de anomalías

✅ Cómo evaluar y comparar modelos

✅ Visualización de resultados con PCA

✅ Análisis detallado por tipo de ataque

### 🌟 ¡Excelente trabajo!

Ahora tienes las herramientas para aplicar detección de anomalías en tus propios proyectos de ciberseguridad.

---